In [3]:
import pandas as pd, re
from urllib.parse import quote
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL

df = pd.read_csv("extracted_knowledge.csv")

CIN = Namespace("http://cinema-kb.org/")
g = Graph()
g.bind("cin", CIN)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

TYPE_MAP = {
    "PERSON":       CIN.Person,
    "ORG":          CIN.Organization,
    "GPE":          CIN.Place,
    "WORK_OF_ART":  CIN.Film,
    "DATE":         CIN.Date,
}

def sanitize_uri(label: str) -> str:
    label = re.sub(r'["\'`]\.\[?\d+', '', label)   # remove ".[123 artifacts
    label = re.sub(r',["\'`\s]*\[?\d+', '', label) # remove ,[357 artifacts
    label = re.sub(r'["\'\`]', '', label)           # remove remaining quotes
    label = label.replace(':', '_').replace('?', '').replace('#', '')
    label = label.replace('(', '').replace(')', '').replace('/', '_')
    label = label.replace('–', '-').replace('—', '-')
    label = label.strip().replace(' ', '_')
    label = re.sub(r'_+', '_', label).strip('_')
    return quote(label, safe='_-.')                 # percent-encode accents etc.

skipped = 0
for _, row in df.iterrows():
    clean = sanitize_uri(row["entity"])
    if not clean:           # skip if nothing is left after sanitization
        skipped += 1
        continue
    entity_uri = URIRef(CIN + clean)
    rdf_type   = TYPE_MAP.get(row["label"], CIN.Entity)
    g.add((entity_uri, RDF.type, rdf_type))
    g.add((entity_uri, RDFS.label, Literal(row["entity"], lang="en")))  # keep raw label as literal

print(f"Triples: {len(g)} | Skipped (empty after sanitization): {skipped}")
g.serialize("private_kb.ttl", format="turtle")

Triples: 17483 | Skipped (empty after sanitization): 0


<Graph identifier=N899371b26f564235a8e2e8e8917f45cc (<class 'rdflib.graph.Graph'>)>

In [5]:
import requests, time, csv, pandas as pd

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "CinemaKGBot/1.0 (university lab; contact@example.com)"}

def search_wikidata(label, retries=3):
    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "format": "json",
        "limit": 1
    }
    for attempt in range(retries):
        try:
            r = requests.get(WIKIDATA_API, params=params,
                             headers=HEADERS, timeout=10)
            # Guard: empty or non-200 response
            if r.status_code != 200 or not r.text.strip():
                print(f"[WARN] Empty/bad response for '{label}' "
                      f"(status {r.status_code}), retrying...")
                time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
                continue
            results = r.json().get("search", [])
            if results:
                return (results[0]["id"],
                        results[0].get("description", ""),
                        results[0].get("match", {}).get("text", ""))
            return None, None, None
        except Exception as e:
            print(f"[ERROR] '{label}': {e}, retrying...")
            time.sleep(2 ** attempt)
    print(f"[FAILED] '{label}' after {retries} attempts.")
    return None, None, None

# Only align non-DATE entities
df = pd.read_csv("extracted_knowledge.csv")
entities = df[df["label"] != "DATE"]["entity"].unique()

alignment = []
for i, ent in enumerate(entities):
    qid, desc, matched = search_wikidata(ent)
    confidence = 0.95 if matched and matched.lower() == ent.lower() \
            else 0.70 if qid else 0.0
    alignment.append({
        "private_entity": ent,
        "wikidata_uri": f"wd:{qid}" if qid else "NOT_FOUND",
        "description": desc,
        "confidence": confidence
    })
    # Progress log every 50 entities
    if i % 50 == 0:
        print(f"[{i}/{len(entities)}] Last: '{ent}' → {qid or 'NOT_FOUND'}")
    time.sleep(0.3)  # polite delay between requests

align_df = pd.DataFrame(alignment)
align_df.to_csv("alignment.csv", index=False)
print(f"\nDone. {len(align_df)} entities processed.")
print(align_df["wikidata_uri"].apply(lambda x: x != "NOT_FOUND").sum(),
      "successfully aligned.")

[0/6194] Last: 'Stanley Kubrick' → Q2001
[50/6194] Last: 'the United States Chess Federation' → NOT_FOUND
[100/6194] Last: 'Vincent Cartier' → NOT_FOUND
[150/6194] Last: 'Hayden' → Q16841766
[200/6194] Last: 'Humbert' → Q9294936
[250/6194] Last: 'Beethoven' → Q255
[300/6194] Last: 'Symphonie fantastique' → Q462313
[350/6194] Last: 'John Williams's' → Q1670253
[400/6194] Last: 'Anatole Litvak' → Q213581
[450/6194] Last: 'György Ligeti' → Q154331
[500/6194] Last: 'Orson Welles' → Q24829
[550/6194] Last: 'Filmworker' → Q39926605
[600/6194] Last: 'the Wayback Machine' → Q110087414
[650/6194] Last: 'Los Angeles Times' → Q188515
[700/6194] Last: 'Rick' → Q18073734
[750/6194] Last: 'Kubrick's The Shining – The Opening' → NOT_FOUND
[800/6194] Last: '2001: A Space Odyssey: New Essays by Robert Kolker' → NOT_FOUND
[850/6194] Last: 'Estrin' → Q21451952
[900/6194] Last: 'Stanley Kubrick: on' → NOT_FOUND
[950/6194] Last: 'Cinema of Stanley Kubrick: Third Edition' → NOT_FOUND
[1000/6194] Last: 'McFa

In [7]:
import re
from urllib.parse import quote
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL

CIN = Namespace("http://cinema-kb.org/")
WD  = Namespace("http://www.wikidata.org/entity/")

def sanitize_uri(label: str) -> str:
    """Aggressively clean any string into a valid URI local name."""
    # Remove ALL types of quotes (straight, curly, backtick)
    label = re.sub(r'["\'\u2018\u2019\u201c\u201d`]', '', label)
    # Remove citation artifacts like .[146 or ,[357
    label = re.sub(r'\.\[?\d+', '', label)
    label = re.sub(r',\s*\[?\d+', '', label)
    # Replace URI-breaking characters
    label = label.replace(':', '_').replace('#', '').replace('?', '')
    label = label.replace('(', '').replace(')', '').replace('/', '_')
    label = label.replace('–', '-').replace('—', '-').replace('&', 'and')
    label = label.replace('<', '').replace('>', '').replace('|', '')
    label = label.replace('^', '').replace('\\', '').replace('{', '').replace('}', '')
    # Normalise spaces and underscores
    label = label.strip().replace(' ', '_')
    label = re.sub(r'_+', '_', label).strip('_')
    # Percent-encode anything remaining (accents, etc.)
    return quote(label, safe='_-.')

# ── Rebuild the graph cleanly from scratch ───────────────────────────────────
g = Graph()
g.bind("cin", CIN)
g.bind("wd",  WD)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

TYPE_MAP = {
    "PERSON":       CIN.Person,
    "ORG":          CIN.Organization,
    "GPE":          CIN.Place,
    "WORK_OF_ART":  CIN.Film,
    "DATE":         CIN.Date,
}

# Step 1 — add all entities
import pandas as pd
df = pd.read_csv("extracted_knowledge.csv")

skipped = 0
for _, row in df.iterrows():
    clean = sanitize_uri(row["entity"])
    if not clean:
        skipped += 1
        continue
    entity_uri = URIRef(CIN + clean)
    g.add((entity_uri, RDF.type,      TYPE_MAP.get(row["label"], CIN.Entity)))
    g.add((entity_uri, RDFS.label,    Literal(row["entity"], lang="en")))

print(f"After Step 1 — Triples: {len(g)} | Skipped: {skipped}")

# Step 2 — add owl:sameAs alignment triples
align_df = pd.read_csv("alignment.csv")

for _, row in align_df.iterrows():
    if row["confidence"] >= 0.90 and row["wikidata_uri"] != "NOT_FOUND":
        clean        = sanitize_uri(row["private_entity"])   # <-- sanitize here too
        if not clean:
            continue
        private_uri  = URIRef(CIN + clean)
        wikidata_uri = URIRef(WD  + row["wikidata_uri"].replace("wd:", ""))
        g.add((private_uri, OWL.sameAs, wikidata_uri))

print(f"After Step 2 — Triples: {len(g)}")

# Serialize
g.serialize("private_kb_aligned.ttl", format="turtle")
print("Saved to private_kb_aligned.ttl")

After Step 1 — Triples: 17331 | Skipped: 0
After Step 2 — Triples: 21182
Saved to private_kb_aligned.ttl


In [11]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL

# These should already exist from previous cells — redefine if needed
CIN = Namespace("http://cinema-kb.org/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

g.bind("wdt", WDT)
g.bind("cin", CIN)

PREDICATE_ALIGNMENT = {
    "direct":  ("P57",  "director",       OWL.equivalentProperty),
    "produce": ("P162", "producer",       OWL.equivalentProperty),
    "earn":    ("P166", "award received", OWL.equivalentProperty),
    "star":    ("P161", "cast member",    OWL.equivalentProperty),
    "write":   ("P58",  "screenwriter",   OWL.equivalentProperty),
}

for private_pred, (wdt_id, label, alignment_type) in PREDICATE_ALIGNMENT.items():
    private_uri  = URIRef(CIN + private_pred)
    wikidata_uri = URIRef(WDT + wdt_id)
    g.add((private_uri, RDF.type,       OWL.ObjectProperty))
    g.add((private_uri, RDFS.label,     Literal(private_pred, lang="en")))
    g.add((private_uri, alignment_type, wikidata_uri))

g.serialize("ontology.ttl", format="turtle")
print(f"Predicate alignment saved. Total triples in graph: {len(g)}")

Predicate alignment saved. Total triples in graph: 21197


In [12]:
import time
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", "CinemaKGBot/1.0 (university lab)")

# Wikidata QIDs for our 8 core cinema entities
CORE_ENTITIES = {
    "Stanley_Kubrick":        "Q2001",
    "Christopher_Nolan":      "Q25191",
    "Martin_Scorsese":        "Q41148",
    "Akira_Kurosawa":         "Q8006",
    "The_Godfather":          "Q47703",
    "Pulp_Fiction":           "Q104123",
    "Inception":              "Q25188",
    "2001_A_Space_Odyssey":   "Q103474",
}

def expand_entity(qid, limit=1000):
    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{qid} ?p ?o .
      FILTER(!isLiteral(?o) || LANG(?o) = "en")
    }} LIMIT {limit}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.query().convert()
        return [(r["p"]["value"], r["o"]["value"])
                for r in results["results"]["bindings"]]
    except:
        return []

expanded_triples = []
for name, qid in CORE_ENTITIES.items():
    triples = expand_entity(qid)
    for p, o in triples:
        expanded_triples.append((f"wd:{qid}", p, o))
    print(f"[{name}] {len(triples)} triples fetched")
    time.sleep(1)

print(f"\nTotal expanded triples: {len(expanded_triples)}")

[Stanley_Kubrick] 445 triples fetched
[Christopher_Nolan] 314 triples fetched
[Martin_Scorsese] 580 triples fetched
[Akira_Kurosawa] 492 triples fetched
[The_Godfather] 565 triples fetched
[Pulp_Fiction] 630 triples fetched
[Inception] 392 triples fetched
[2001_A_Space_Odyssey] 447 triples fetched

Total expanded triples: 3865


In [14]:
def expand_predicate(wdt_prop, limit=20000):
    """Fetch all triples sharing a cinema-relevant predicate."""
    query = f"""
    SELECT ?s ?o WHERE {{
      ?s wdt:{wdt_prop} ?o .
      # Restrict to films
      ?s wdt:P31 wd:Q11424 .
    }} LIMIT {limit}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.query().convert()
        return [(r["s"]["value"], f"wdt:{wdt_prop}", r["o"]["value"])
                for r in results["results"]["bindings"]]
    except:
        return []

# Expand on key cinema predicates
CINEMA_PROPS = {
    "P57":  "director",
    "P161": "cast member",
    "P162": "producer",
    "P166": "award received",
    "P364": "original language",
    "P495": "country of origin",
    "P577": "publication date",
}

for prop, label in CINEMA_PROPS.items():
    triples = expand_predicate(prop, limit=10000)
    expanded_triples.extend(triples)
    print(f"[{label}] +{len(triples)} triples → total: {len(expanded_triples)}")
    time.sleep(2)

[director] +10000 triples → total: 83865
[cast member] +10000 triples → total: 93865
[producer] +10000 triples → total: 103865
[award received] +10000 triples → total: 113865
[original language] +10000 triples → total: 123865
[country of origin] +10000 triples → total: 133865
[publication date] +10000 triples → total: 143865


In [15]:
# Deduplicate
expanded_triples = list(set(expanded_triples))

# Add to graph and serialize
for s, p, o in expanded_triples:
    g.add((URIRef(s), URIRef(p), URIRef(o) if o.startswith("http") else Literal(o)))

# Remove isolated nodes (connectivity check)
subjects   = set(g.subjects())
objects    = set(g.objects())
all_nodes  = subjects | objects
connected  = subjects & objects  # nodes that are both subject and object
print(f"Total triples:   {len(g)}")
print(f"Total entities:  {len(all_nodes)}")
print(f"Connected nodes: {len(connected)}")

# Export deliverables
g.serialize("expanded_kb.nt",      format="nt")       # for KGE
g.serialize("expanded_kb.ttl",     format="turtle")    # human-readable
g.serialize("ontology.ttl",        format="turtle")    # ontology only
align_df.to_csv("alignment.csv",   index=False)        # alignment table

Total triples:   99106
Total entities:  66864
Connected nodes: 178


d:\anaconda\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [16]:
print("=" * 40)
print("FINAL KB STATISTICS")
print("=" * 40)
print(f"Total triples:    {len(g)}")
print(f"Total entities:   {len(all_nodes)}")
print(f"Total relations:  {len(set(g.predicates()))}")
print(f"Aligned entities: {len(align_df[align_df['wikidata_uri'] != 'NOT_FOUND'])}")
print(f"Alignment rate:   {len(align_df[align_df['wikidata_uri'] != 'NOT_FOUND']) / len(align_df):.1%}")
print("=" * 40)
print("Files saved:")
print("  - expanded_kb.nt   (for KGE)")
print("  - expanded_kb.ttl  (human-readable)")
print("  - ontology.ttl     (predicates)")
print("  - alignment.csv    (entity mapping)")

FINAL KB STATISTICS
Total triples:    99106
Total entities:   66864
Total relations:  857
Aligned entities: 4403
Alignment rate:   71.1%
Files saved:
  - expanded_kb.nt   (for KGE)
  - expanded_kb.ttl  (human-readable)
  - ontology.ttl     (predicates)
  - alignment.csv    (entity mapping)
